# Data Preprocessing

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif

from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LassoCV

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

import joblib

In [2]:
df = pd.read_csv(
    "../data/pcos_dataset.csv"
)

df.head()

,Age,Height_cm,Weight_kg,Family_History_PCOS,Regular_Menstrual_Cycles,Average_Cycle_Length_Days,Miss_or_Delay_Periods,Bleeding_Duration_Days,Painful_Periods,Recent_Weight_Gain,...,Stress_Level_Rating,Mood_Swings,Anxiety_or_Low_Mood,Health_Concern,Doctor_Diagnosed_PCOS,Diabetes_or_Prediabetes,Thyroid_Disorder,Hormonal_Medication,Infertility_Problem,PCOS_Target
0,21,155.6,72.2,no,yes,24,rarely,4,mild,yes,...,4,no,sometimes,no,no,no,no,no,yes,no
1,35,164.8,60.9,no,yes,29,rarely,6,moderate,no,...,2,no,sometimes,no,no,no,no,no,not applicable,no
2,27,166.2,57.5,no,yes,23,sometimes,5,not at all,no,...,4,yes,often,yes,no,no,no,no,not applicable,no
3,19,157.5,51.9,yes,yes,34,sometimes,4,mild,no,...,3,yes,never,no,no,no,no,no,no,no
4,40,157.4,52.9,no,yes,21,never,3,mild,no,...,3,yes,sometimes,yes,no,no,no,yes,not applicable,no


In [3]:
print(df.shape)

print(df.columns.tolist())

(1800, 30)
['Age', 'Height_cm', 'Weight_kg', 'Family_History_PCOS', 'Regular_Menstrual_Cycles', 'Average_Cycle_Length_Days', 'Miss_or_Delay_Periods', 'Bleeding_Duration_Days', 'Painful_Periods', 'Recent_Weight_Gain', 'Acne_or_Pimples', 'Hair_Fall_or_Thinning', 'Dark_Patches', 'Difficult_to_Lose_Weight', 'Sugary_Food_Cravings', 'Daytime_Fatigue', 'Exercise_Frequency', 'Fast_Food_Frequency', 'Sleep_Hours', 'Sleep_Quality_Rating', 'Stress_Level_Rating', 'Mood_Swings', 'Anxiety_or_Low_Mood', 'Health_Concern', 'Doctor_Diagnosed_PCOS', 'Diabetes_or_Prediabetes', 'Thyroid_Disorder', 'Hormonal_Medication', 'Infertility_Problem', 'PCOS_Target']


In [4]:
df.columns = [

"Age",
"Height_cm",
"Weight_kg",

"Family_History_PCOS",

"Regular_Menstrual_Cycles",

"Cycle_Length",

"Miss_or_Delay_Periods",

"Bleeding_Duration",

"Painful_Periods",

"Recent_Weight_Gain",

"Acne_or_Pimples",

"Hair_Fall_or_Thinning",

"Dark_Patches",

"Difficult_to_Lose_Weight",

"Sugary_Food_Cravings",

"Daytime_Fatigue",

"Exercise_Frequency",

"Fast_Food_Frequency",

"Sleep_Hours",

"Sleep_Quality",

"Stress_Level",

"Mood_Swings",

"Anxiety_or_Low_Mood",

"Health_Concern",

"Doctor_Diagnosed_PCOS",

"Diabetes_or_Prediabetes",

"Thyroid_Disorder",

"Hormonal_Medication",

"Infertility_Problem",

"PCOS"

]

In [5]:
print(
    df["PCOS"].unique()
)

['no' 'yes']


In [6]:
df["PCOS"] = df["PCOS"].replace({

    "yes":1,
    "no":0,
})

df["PCOS"] = df["PCOS"].astype(int)

C:\Users\User\AppData\Local\Temp\ipykernel_19204\530540760.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["PCOS"] = df["PCOS"].replace({


In [7]:
for col in df.columns:

    if df[col].dtype == "object" or str(df[col].dtype) == "string":

        if col != "PCOS":

            df[col] = pd.factorize(
                df[col]
            )[0]

In [8]:
print(
    df.select_dtypes(
        include=["object","string"]
    ).columns
)

Index([], dtype='object')


In [9]:
X = df.drop(
    columns=[
        "PCOS",
        "Doctor_Diagnosed_PCOS"
    ]
)

y = df["PCOS"]

print(X.shape)
print(y.shape)

(1800, 28)
(1800,)


In [10]:
print(y.value_counts())

PCOS
0    1120
1     680
Name: count, dtype: int64


In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(1440, 28)
(360, 28)


In [12]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for col in X_train.columns:

    if X_train[col].dtype == "object" or str(X_train[col].dtype) == "string":

        X_train[col] = le.fit_transform(
            X_train[col]
        )

        X_test[col] = le.transform(
            X_test[col]
        )

In [13]:
print(
    X_train.select_dtypes(
        include=["object", "string"]
    ).columns
)

Index([], dtype='object')


In [14]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train
)

X_test_scaled = scaler.transform(
    X_test
)

In [15]:
joblib.dump(
    scaler,
    "../models/scaler.pkl"
)

['../models/scaler.pkl']

### Baseline Logistic Regression (All Features)

In [16]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=5000
)

model.fit(
    X_train_scaled,
    y_train
)

y_pred = model.predict(
    X_test_scaled
)

In [17]:
from sklearn.metrics import accuracy_score

accuracy_score(
    y_test,
    y_pred
)

0.7861111111111111

In [18]:
print(X_train.shape)
print(X_test.shape)
accuracy_score(y_test, y_pred)

(1440, 28)
(360, 28)


0.7861111111111111

# Feature Selection

In [19]:
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif

anova = SelectKBest(
    score_func=f_classif,
    k=15
)

anova.fit(
    X_train_scaled,
    y_train
)

anova_scores = pd.DataFrame({

    "Feature": X.columns,

    "Score": anova.scores_

})

anova_scores = anova_scores.sort_values(
    by="Score",
    ascending=False
)

anova_scores.head(15)

,Feature,Score
4,Regular_Menstrual_Cycles,145.442934
3,Family_History_PCOS,120.981784
2,Weight_kg,97.304652
9,Recent_Weight_Gain,58.979620
5,Cycle_Length,52.743879
12,Dark_Patches,46.305623
10,Acne_or_Pimples,36.926348
11,Hair_Fall_or_Thinning,25.824861
13,Difficult_to_Lose_Weight,25.122045
24,Diabetes_or_Prediabetes,10.347988


In [20]:
selected_features_anova = X.columns[
    anova.get_support()
]

print(selected_features_anova)

Index(['Height_cm', 'Weight_kg', 'Family_History_PCOS',
       'Regular_Menstrual_Cycles', 'Cycle_Length', 'Miss_or_Delay_Periods',
       'Recent_Weight_Gain', 'Acne_or_Pimples', 'Hair_Fall_or_Thinning',
       'Dark_Patches', 'Difficult_to_Lose_Weight', 'Mood_Swings',
       'Health_Concern', 'Diabetes_or_Prediabetes', 'Infertility_Problem'],
      dtype='object')


In [21]:
from sklearn.linear_model import LogisticRegression

lasso = LogisticRegression(

    penalty="l1",

    solver="liblinear",

    max_iter=5000

)

lasso.fit(
    X_train_scaled,
    y_train
)

lasso_features = pd.DataFrame({

    "Feature": X.columns,

    "Coefficient":
    abs(lasso.coef_[0])

})

lasso_features.sort_values(
    by="Coefficient",
    ascending=False
).head(15)

,Feature,Coefficient
4,Regular_Menstrual_Cycles,1.101638
3,Family_History_PCOS,1.034638
2,Weight_kg,0.884924
5,Cycle_Length,0.661479
9,Recent_Weight_Gain,0.553083
12,Dark_Patches,0.552705
11,Hair_Fall_or_Thinning,0.546574
10,Acne_or_Pimples,0.517269
13,Difficult_to_Lose_Weight,0.498835
1,Height_cm,0.331815


In [22]:
selected_features_lasso = lasso_features.sort_values(
    by="Coefficient",
    ascending=False
)["Feature"].head(15)

print(selected_features_lasso)

4     Regular_Menstrual_Cycles
3          Family_History_PCOS
2                    Weight_kg
5                 Cycle_Length
9           Recent_Weight_Gain
12                Dark_Patches
11       Hair_Fall_or_Thinning
10             Acne_or_Pimples
13    Difficult_to_Lose_Weight
1                    Height_cm
27         Infertility_Problem
6        Miss_or_Delay_Periods
24     Diabetes_or_Prediabetes
21                 Mood_Swings
8              Painful_Periods
Name: Feature, dtype: object


In [23]:
from sklearn.ensemble import RandomForestClassifier
import pandas as pd

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(
    X_train_scaled,
    y_train
)

rf_features = pd.DataFrame({

    "Feature": X.columns,

    "Importance": rf.feature_importances_

})

rf_features = rf_features.sort_values(
    by="Importance",
    ascending=False
)

rf_features.head(15)

,Feature,Importance
2,Weight_kg,0.118723
5,Cycle_Length,0.089928
4,Regular_Menstrual_Cycles,0.076145
1,Height_cm,0.070611
3,Family_History_PCOS,0.064242
18,Sleep_Hours,0.059993
6,Miss_or_Delay_Periods,0.058505
0,Age,0.051841
7,Bleeding_Duration,0.034003
9,Recent_Weight_Gain,0.031610


In [24]:
selected_features_rf = rf_features[
    "Feature"
].head(15)

print(selected_features_rf)

2                    Weight_kg
5                 Cycle_Length
4     Regular_Menstrual_Cycles
1                    Height_cm
3          Family_History_PCOS
18                 Sleep_Hours
6        Miss_or_Delay_Periods
0                          Age
7            Bleeding_Duration
9           Recent_Weight_Gain
20                Stress_Level
19               Sleep_Quality
22         Anxiety_or_Low_Mood
8              Painful_Periods
12                Dark_Patches
Name: Feature, dtype: object


In [25]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    max_iter=5000
)

rfe = RFE(

    estimator=lr,

    n_features_to_select=15

)

rfe.fit(
    X_train_scaled,
    y_train
)

selected_features_rfe = X.columns[
    rfe.support_
]

print(selected_features_rfe)

Index(['Height_cm', 'Weight_kg', 'Family_History_PCOS',
       'Regular_Menstrual_Cycles', 'Cycle_Length', 'Miss_or_Delay_Periods',
       'Painful_Periods', 'Recent_Weight_Gain', 'Acne_or_Pimples',
       'Hair_Fall_or_Thinning', 'Dark_Patches', 'Difficult_to_Lose_Weight',
       'Mood_Swings', 'Diabetes_or_Prediabetes', 'Infertility_Problem'],
      dtype='object')


In [26]:
pip install pyswarms==1.3.0

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [27]:
combined_features = [

    "Height_cm",
    "Weight_kg",
    "Family_History_PCOS",
    "Regular_Menstrual_Cycles",
    "Cycle_Length",
    "Miss_or_Delay_Periods",
    "Recent_Weight_Gain",
    "Acne_or_Pimples",
    "Hair_Fall_or_Thinning",
    "Dark_Patches",
    "Difficult_to_Lose_Weight",
    "Mood_Swings",
    "Diabetes_or_Prediabetes",
    "Infertility_Problem",
    "Painful_Periods",

    "Age",
    "Bleeding_Duration",
    "Sleep_Hours",
    "Sleep_Quality",
    "Stress_Level",
    "Anxiety_or_Low_Mood"

]
len(combined_features)

21

In [28]:
X_bpso = X[combined_features]

print(X_bpso.shape)

(1800, 21)


In [29]:
from sklearn.model_selection import train_test_split

X_train_bpso, X_test_bpso, y_train_bpso, y_test_bpso = train_test_split(

    X_bpso,
    y,

    test_size=0.2,

    random_state=42,

    stratify=y

)

In [30]:
from sklearn.preprocessing import StandardScaler

scaler_bpso = StandardScaler()

X_train_bpso_scaled = scaler_bpso.fit_transform(
    X_train_bpso
)

X_test_bpso_scaled = scaler_bpso.transform(
    X_test_bpso
)

In [31]:
import numpy as np

from pyswarms.discrete.binary import BinaryPSO

from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import cross_val_score

In [32]:
def fitness_function(particles):

    n_particles = particles.shape[0]

    scores = []

    for particle in particles:

        if np.count_nonzero(particle) == 0:

            scores.append(1)

            continue

        selected_columns = np.where(
            particle == 1
        )[0]

        X_selected = X_train_bpso_scaled[
            :,
            selected_columns
        ]

        model = LogisticRegression(
            max_iter=5000
        )

        accuracy = cross_val_score(

            model,

            X_selected,

            y_train_bpso,

            cv=5,

            scoring="accuracy"

        ).mean()

        scores.append(
            1 - accuracy
        )

    return np.array(scores)

In [33]:
options = {

    'c1': 1.5,

    'c2': 1.5,

    'w': 0.7,

    'k': 10,

    'p': 2

}

In [34]:
optimizer = BinaryPSO(

    n_particles=30,

    dimensions=X_train_bpso_scaled.shape[1],

    options=options

)

In [35]:
best_cost, best_position = optimizer.optimize(

    fitness_function,

    iters=30

)

2026-06-11 20:11:42,322 - pyswarms.discrete.binary - INFO - Optimize for 30 iters with {'c1': 1.5, 'c2': 1.5, 'w': 0.7, 'k': 10, 'p': 2}
pyswarms.discrete.binary: 100%|████████████████████████████████████████████████████████████████████|30/30, best_cost=0.196
2026-06-11 20:12:11,164 - pyswarms.discrete.binary - INFO - Optimization finished | best cost: 0.1958333333333332, best pos: [0 1 1 1 0 1 1 1 1 1 1 0 1 1 0 0 0 1 1 0 1]


In [36]:
selected_features_bpso = np.array(

    combined_features

)[

    best_position.astype(bool)

]

print(selected_features_bpso)

['Weight_kg' 'Family_History_PCOS' 'Regular_Menstrual_Cycles'
 'Miss_or_Delay_Periods' 'Recent_Weight_Gain' 'Acne_or_Pimples'
 'Hair_Fall_or_Thinning' 'Dark_Patches' 'Difficult_to_Lose_Weight'
 'Diabetes_or_Prediabetes' 'Infertility_Problem' 'Sleep_Hours'
 'Sleep_Quality' 'Anxiety_or_Low_Mood']


In [37]:
print(best_cost)

0.1958333333333332


In [38]:
accuracy = 1 - best_cost

print(accuracy)

0.8041666666666668


In [39]:
X_final = X[selected_features_bpso]

print(X_final.shape)

(1800, 14)


In [40]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(

    X_final,

    y,

    test_size=0.2,

    random_state=42,

    stratify=y

)

In [41]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train
)

X_test_scaled = scaler.transform(
    X_test
)

In [42]:
import os

os.makedirs(
    "../models",
    exist_ok=True
)

In [43]:
import joblib

joblib.dump(

    scaler,

    "../models/scaler.pkl"

)

['../models/scaler.pkl']

# Model Training

In [44]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression()

lr.fit(

    X_train_scaled,

    y_train

)

lr_pred = lr.predict(

    X_test_scaled

)

In [45]:
from sklearn.metrics import (

    accuracy_score,

    precision_score,

    recall_score,

    f1_score

)

print(

    "Accuracy:",

    accuracy_score(
        y_test,
        lr_pred
    )

)

print(

    "Precision:",

    precision_score(
        y_test,
        lr_pred
    )

)

print(

    "Recall:",

    recall_score(
        y_test,
        lr_pred
    )

)

print(

    "F1:",

    f1_score(
        y_test,
        lr_pred
    )

)

Accuracy: 0.7694444444444445
Precision: 0.7086614173228346
Recall: 0.6617647058823529
F1: 0.6844106463878327


In [46]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(

    n_estimators=200,

    random_state=42

)

rf.fit(

    X_train,

    y_train

)

rf_pred = rf.predict(
    X_test
)

In [47]:
print(

    "Accuracy:",

    accuracy_score(
        y_test,
        rf_pred
    )

)

print(

    "Precision:",

    precision_score(
        y_test,
        rf_pred
    )

)

print(

    "Recall:",

    recall_score(
        y_test,
        rf_pred
    )

)

print(

    "F1:",

    f1_score(
        y_test,
        rf_pred
    )

)

Accuracy: 0.7694444444444445
Precision: 0.7304347826086957
Recall: 0.6176470588235294
F1: 0.6693227091633466


In [48]:
from sklearn.svm import SVC

svm = SVC(

    kernel="rbf",

    probability=True

)

svm.fit(

    X_train_scaled,

    y_train

)

svm_pred = svm.predict(

    X_test_scaled

)

In [49]:
print(

    "Accuracy:",

    accuracy_score(
        y_test,
        svm_pred
    )

)

print(

    "Precision:",

    precision_score(
        y_test,
        svm_pred
    )

)

print(

    "Recall:",

    recall_score(
        y_test,
        svm_pred
    )

)

print(

    "F1:",

    f1_score(
        y_test,
        svm_pred
    )

)

Accuracy: 0.7583333333333333
Precision: 0.696
Recall: 0.6397058823529411
F1: 0.6666666666666666


In [50]:
from xgboost import XGBClassifier

In [51]:
from xgboost import XGBClassifier

xgb = XGBClassifier(

    random_state=42

)

xgb.fit(

    X_train,

    y_train

)

xgb_pred = xgb.predict(
    X_test
)

In [52]:
print(

    "Accuracy:",

    accuracy_score(
        y_test,
        xgb_pred
    )

)

print(

    "Precision:",

    precision_score(
        y_test,
        xgb_pred
    )

)

print(

    "Recall:",

    recall_score(
        y_test,
        xgb_pred
    )

)

print(

    "F1:",

    f1_score(
        y_test,
        xgb_pred
    )

)

Accuracy: 0.7666666666666667
Precision: 0.7280701754385965
Recall: 0.6102941176470589
F1: 0.664


# Model Comparison

In [59]:
import pandas as pd

results = pd.DataFrame({

    "Model":[

        "Logistic Regression",

        "Random Forest",

        "SVM",

        "XGBoost"

    ],

    "Accuracy":[

        accuracy_score(
            y_test,
            lr_pred
        ),

        accuracy_score(
            y_test,
            rf_pred
        ),

        accuracy_score(
            y_test,
            svm_pred
        ),

        accuracy_score(
            y_test,
            xgb_pred
        )

    ]

})

results.sort_values(

    by="Accuracy",

    ascending=False

)

,Model,Accuracy
0,Logistic Regression,0.769444
1,Random Forest,0.769444
3,XGBoost,0.766667
2,SVM,0.758333


In [61]:
import pandas as pd

results = pd.DataFrame({

    "Model": [
        "Logistic Regression",
        "Random Forest",
        "SVM",
        "XGBoost"
    ],

    "Accuracy": [
        0.7694444444444445,
        0.7694444444444445,
        0.7583333333333333,
        0.7666666666666667
    ],

    "Precision": [
        0.7086614173228346,
        0.7304347826086957,
        0.696,
        0.7280701754385965
    ],

    "Recall": [
        0.6617647058823529,
        0.6176470588235294,
        0.6397058823529411,
        0.6102941176470589
    ],

    "F1 Score": [
        0.6844106463878327,
        0.6693227091633466,
        0.6666666666666666,
        0.664
    ]

})

results

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.769444,0.708661,0.661765,0.684411
1,Random Forest,0.769444,0.730435,0.617647,0.669323
2,SVM,0.758333,0.696000,0.639706,0.666667
3,XGBoost,0.766667,0.728070,0.610294,0.664000


In [62]:
results.sort_values(
    by="F1 Score",
    ascending=False
)

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.769444,0.708661,0.661765,0.684411
1,Random Forest,0.769444,0.730435,0.617647,0.669323
2,SVM,0.758333,0.696000,0.639706,0.666667
3,XGBoost,0.766667,0.728070,0.610294,0.664000


In [63]:
results.sort_values(
    by="Recall",
    ascending=False
)

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.769444,0.708661,0.661765,0.684411
2,SVM,0.758333,0.696000,0.639706,0.666667
1,Random Forest,0.769444,0.730435,0.617647,0.669323
3,XGBoost,0.766667,0.728070,0.610294,0.664000


# Best Model Selection

In [64]:
best_model = lr

In [65]:
import joblib

joblib.dump(
    best_model,
    "../models/model.pkl"
)

['../models/model.pkl']

In [66]:
joblib.dump(
    scaler,
    "../models/scaler.pkl"
)

['../models/scaler.pkl']

In [67]:
import os

print(
    os.listdir("../models")
)

['.ipynb_checkpoints', 'features.pkl', 'model.pkl', 'scaler.pkl']


In [68]:
joblib.dump(
    selected_features_bpso,
    "../models/features.pkl"
)

['../models/features.pkl']

In [3]:
import joblib

features = joblib.load("../models/features.pkl")

print("TYPE:")
print(type(features))

print("\nFEATURES:")
print(features)

TYPE:
<class 'numpy.ndarray'>

FEATURES:
['Weight_kg' 'Family_History_PCOS' 'Regular_Menstrual_Cycles'
 'Miss_or_Delay_Periods' 'Recent_Weight_Gain' 'Acne_or_Pimples'
 'Hair_Fall_or_Thinning' 'Dark_Patches' 'Difficult_to_Lose_Weight'
 'Diabetes_or_Prediabetes' 'Infertility_Problem' 'Sleep_Hours'
 'Sleep_Quality' 'Anxiety_or_Low_Mood']
